In [1]:
from llama_index.packs.tables.chain_of_table.base import (
    ChainOfTableQueryEngine,
    serialize_table,
)
from openai import OpenAI

import sys
sys.path.append('../')

from utils.MyUtils import MyPipeline
from utils.MeLogSingle import MeLogger
from utils.MyPreprocessing import PreprocessingDatasets

In [2]:
class LLMS:
    def __init__(self, datasets:dict):
        # self._logger = MeLogger()
        self._prep = PreprocessingDatasets()
        self.datasets = datasets
        self.german = self.pre_processing_german_credit()
        self.law = self.pre_processing_law()
        self.pima = self.pre_processing_pima()
        self.npha = self.pre_processing_npha()
        self.cirohis = self.pre_processing_cirohis()

    def pre_processing_german_credit(self):
        german_credit_df = self.datasets['german'].copy()

        map_gender = {"A91":"male",
                      "A92":"female",
                      "A93":"male",
                      "A94":"male",
                      "A95":"female"}
        
        german_credit_df["personal-status-and-sex"] = german_credit_df["personal-status-and-sex"].map(map_gender)

        german_credit_df = self._prep.ordinal_encoder(german_credit_df, ["age",
                                                                         "checking-account",
                                                                         "savings-account",
                                                                         "employment-since",
                                                                         "telephone",
                                                                         "foreign-worker",
                                                                         "personal-status-and-sex"])
        german_credit_df = self._prep.label_encoder(german_credit_df, ["target"])
        
        german_credit_df = self._prep.one_hot_encode(german_credit_df, ["credit-history",
                                                                        "purpose",
                                                                        "other-debtors",
                                                                        "property",
                                                                        "other-installment",
                                                                        "housing", 
                                                                        "job"])

        return german_credit_df
    
    def pre_processing_law(self):
        law_df = self.datasets["law_school_clean"].copy()
        law_df = self._prep.label_encoder(law_df, ["target"])
        law_df = self._prep.one_hot_encode(law_df, ["fam_inc", 
                                                    "race"])

        return law_df
    
    def pre_processing_pima(self):
        pima_diabetes_df = self.datasets["pima_diabetes"].copy()
        return pima_diabetes_df
    
    def pre_processing_npha(self):
        npha_doctor_visits_df = self.datasets["NPHA-doctor-visits"].copy()

        npha_doctor_visits_df = self._prep.one_hot_encode(npha_doctor_visits_df, ["Gender",
                                                                            "Race",
                                                                            "Prescription Sleep Medication",
                                                                            "Employment"])

        npha_doctor_visits_df["target"] = npha_doctor_visits_df["target"].astype("int64")
        return npha_doctor_visits_df
    
    def pre_processing_cirohis(self):
        cirrhosis_df = self.datasets['cirrhosis'].copy()
        cirrhosis_df = cirrhosis_df.dropna().drop(columns="ID")
        cirrhosis_df = self._prep.label_encoder(cirrhosis_df, ["target",
                                                         "Sex",
                                                         "Ascites",
                                                         "Hepatomegaly",
                                                         "Spiders",
                                                         "Edema"])
        cirrhosis_df = self._prep.ordinal_encoder(cirrhosis_df, ["Stage"])
        cirrhosis_df = self._prep.one_hot_encode(cirrhosis_df, ["Drug"]).dropna()

        return cirrhosis_df
    
    def cria_tabela_llms(self):
        tabela_resultados = {}

        tabela_resultados["datasets"] = [self.pima,
                                        self.npha,
                                        self.cirohis,
                                        self.german,
                                        self.law]
            
        tabela_resultados["nome_datasets"] = ["pima",
                                            "npha",
                                            "cirohis",
                                            "german",
                                            "law"]
            
        tabela_resultados["missing_rate"] = [10,20,40, 60]

        return tabela_resultados

In [8]:
def get_code_completion(client, messages, max_tokens=512, model="Meta-Llama-3-8B-Instruct.Q5_K_M"):
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        max_tokens=max_tokens,
        stop=[
            "<step>"
        ],
        frequency_penalty=1,
        presence_penalty=1,
        top_p=0.7,
        n=10,
        temperature=0.7
    )

    return chat_completion

In [4]:
diretorio = "C:\\Users\\Mult-e\\Desktop\\@Codigos\\MestradoCodigos\\MestradoCodigos\\LLMs\\DatasetsLLMs"
datasets = MyPipeline.carrega_datasets(diretorio)

llms = LLMS(datasets)
tabela_llms = llms.cria_tabela_llms()

client = OpenAI(base_url="http://127.0.0.1:8080/v1",
                api_key = "sk-no-key-required")

## Few Shot examples

In [5]:
FEW_SHOT_PROMPT_1 = """
You are given a Pandas dataframe named students_df:
- Patient: 0 years, 1 car, 2 childs
User's Question: What can be the approximated value of childs for this patient? 
"""
FEW_SHOT_ANSWER_1 = """
result = The approximated value of childs for this patient is 2
"""

FEW_SHOT_PROMPT_2 = """
You are given a Pandas dataframe named students_df:
- Patient: 3 years, 4 car, 5 childs
User's Question: What can be the approximated value of childs for this patient?
"""
FEW_SHOT_ANSWER_2 = """
result = The approximated value of childs for this patient is 5
"""

FEW_SHOT_PROMPT_USER = """
You are given a Pandas dataframe named students_df:
- Patient: 6 years, 7 cars, np.nan childs
User's Question: What can be the approximated value of childs for this patient?
"""

In [6]:
messages = [
    {
        "role": "system",
        "content": "Based on data mentioned, you must have impute the np.nan (missing value) for approximate value for number of childrean for specific patient. Don't create a python code."
    },
    {
        "role": "user",
        "content": FEW_SHOT_PROMPT_1
    },
    {
        "role": "assistant",
        "content": FEW_SHOT_ANSWER_1
    },
    {
        "role": "user",
        "content": FEW_SHOT_PROMPT_2
    },
    {
        "role": "assistant",
        "content": FEW_SHOT_ANSWER_2
    },
    {
        "role": "user",
        "content": FEW_SHOT_PROMPT_USER
    }
]

chat_completion = get_code_completion(client, messages)
            
print(chat_completion.choices[0].message.content)

result = The approximated value of childs for this patient is approximately 8 (assuming a linear relationship between age and number of children)



In [7]:
from mdatagen.multivariate.mMCAR import mMCAR

df = tabela_llms["datasets"][0]

X = df.copy().drop("target",axis=1)
y = df.target.values

gen = mMCAR(X,y)
X = gen.random()

serialized_table = serialize_table(X)

prompt_str = f"""\
[INST] Here's a serialized table.

{serialized_table}

Based on serialized table above, you must have impute the np.nan (missing value) for approximate value.
[\INST]"""

messages = [
    {
        "role": "system",
        "content": "You must have understand the data pattern from user input and impute the missing values on the serialized table.",
    },
    {
        "role": "user",
        "content": prompt_str,
    }
]

chat_completion = get_code_completion(client, messages)
            
print(chat_completion.choices[0].message.content)

Based on the provided table, I will impute missing values using a simple mean or median of other available columns.

**Step 1: Identify rows with missing values**

From the original table, we can see that there are several instances where `INST` and/or `im_end`, `im_start` have np.nan (missing) values. For example:
```python
row 763 : INST is nan
row 764 : im_end is nan
```
**Step 2: Impute missing values using mean or median**

For each column with missing values, we will calculate the mean or median of other available columns and use that as a substitute for the missing value.

1. **INST**: Calculate the mean INST value across all rows.
```python
mean_INST = (93 + 70 + ... + 23) / num_rows
```
Replace np.nan in `row 763` with this calculated mean:
```python
INST[763] = mean_INST
```

2. **im_end**: Calculate the median im_end value across all rows.
```python
median_im_end = sorted([30, 31, ...]) [num_rows // 2]
```
Replace np.nan in `row 764` with this calculated median:
```python
im

1) Um prompt - uma resposta - não imputar cada resposta
2) condesar a informação para construir o prompt **muito importante** - explorar como poderia ser feito
3) informação adicional no prompt para não limitar 

llamafile rodar na VM gpu